In [ ]:
#Importamos las librerías que vamos a utilizar
from IPython.display import HTML
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import shapiro

# Escribimos una función para limpiar los nombres de las columnas, elimina [] y espacios y los reemplaza con _
def limpiar_nombres_columnas(df):
    df = df.copy()
    df.columns = [
        re.sub(r"[\[\]<>]", "", str(col))
        .strip()
        .replace(" ", "_")
        for col in df.columns
    ]
    return df

In [ ]:
ruta = r"Material/Datasets/ai4i2020.csv"
df = pd.read_csv(ruta)

# Aplicamos la limpieza
df = limpiar_nombres_columnas(df)

In [ ]:
# Mostramos la cantidad de filas y columnas del DataFrame
print("El DataFrame tiene", df.shape[0], "filas y", df.shape[1] ,"columnas")


In [ ]:
# Mostramos las primeras filas del DataFrame
df.head()


In [ ]:
# Eliminamos las variables UDI y Product_ID
df.drop(columns=['UDI', 'Product_ID'], inplace=True)

In [ ]:
# Creamos un nuevo DataFrame para juntar informaciones
info_df = pd.DataFrame({
    'Tipo de Dato': df.dtypes,
    'Valores Nulos': df.isnull().sum(),
})

# Asignamos un nombre a la primera columna del DataFrame
info_df = info_df.reset_index().rename(columns={'index': 'Variables'})

info_df


In [ ]:
# Generamos las estadísticas descriptivas del DataFrame
df.describe()


### 🔍 Diagnóstico del Balance de Clases 



In [ ]:
# Vamos a ver la cantidad de fallas que hay en el dataset
conteo_fallas = df['Machine_failure'].value_counts()

# Calculamos el porcentaje de fallas en el dataset
porcentaje_fallas = df['Machine_failure'].value_counts(normalize=True) * 100

# Resumen 
resumen_fallas = pd.DataFrame({
    'Cantidad de fallas' : conteo_fallas,
    'Porcentaje (%)' : porcentaje_fallas.round(4)
})

resumen_fallas.index.name = None

resumen_fallas


A partir del análisis cuantitativo de la variable objetivo `Machine_failure`, se extraen las siguientes conclusiones analíticas fundamentales para el desarrollo del modelo:

* **Predominancia de la Clase Mayoritaria:** El **96.61%** de los registros históricos corresponden a un estado operativo normal (Clase `0`), sumando un total de **9,661 observaciones**.
* **Presencia de la Clase Minoritaria (Target):** Se registraron únicamente **339 eventos de falla mecánica** (Clase `1`), lo que equivale al **3.39%** del conjunto de datos total.


### 🛠️ Análisis Desagregado de Tipos de Falla



In [ ]:
# Creamos una lista con las columnas de fallas específicas
columnas_fallas = ['TWF', 'HDF', 'PWF', 'OSF', 'RNF']

# Sumamos los valores de cada columna
conteo_fallas_especificas = df[columnas_fallas].sum()

# Creamos un DataFrame y lo ordenamos de mayor a menor
tabla_fallas_especificas = pd.DataFrame({
    'Total de Eventos': conteo_fallas_especificas,
}).sort_values(by='Total de Eventos', ascending=False)

# Quitamos el nombre del índice para que quede más limpio
tabla_fallas_especificas.index.name = None

# Calculamos la suma total de eventos y lo agregamos a la tabla
tabla_fallas_especificas.loc['Total'] = tabla_fallas_especificas.sum()

tabla_fallas_especificas


Para entender la naturaleza del desbalance, se aislaron y contabilizaron de forma independiente los modos de falla registrados por los sensores:

* **HDF (Heat Dissipation Failure - 115 eventos):** Es la causa principal de avería. Indica problemas críticos de disipación térmica y sobrecalentamiento en la operación.
* **OSF (Overstrain Failure - 98 eventos):** El segundo factor de riesgo, asociado a tensiones mecánicas excesivas o sobreesfuerzos estructurales.
* **PWF (Power Failure - 95 eventos):** Fallos directamente relacionados con fluctuaciones o pérdidas en el suministro de potencia eléctrica.
* **TWF (Tool Wear Failure - 45 eventos):** Desgaste acumulado de la herramienta de corte que alcanza su límite seguro.
* **RNF (Random Failure - 18 eventos):** Eventos de fallo aleatorios no correlacionados directamente con los sensores principales.
<br>
<br>
> 💡 **Nota de Consistencia de Datos:** Si sumamos todos estos eventos individuales obtenemos 373 fallas, lo cual supera las 339 fallas registradas identificadas con 'Machine_failure = 1'. Este hallazgo es clave para nuestro análisis EDA: **existen ciclos de operación específicos donde la máquina sufrió múltiples modos de falla simultáneos** durante el mecanizado de una pieza.




### Análisis de fallas simultáneas


In [ ]:
fallas_multiples = df[df[columnas_fallas].sum(axis=1) >= 2].copy()

# Agregamos una columna que nos indique que fallas ocurrieron a la misma vez
fallas_multiples['Fallas simultáneas'] = fallas_multiples[columnas_fallas].apply(
    lambda fila: ', '.join([col for col in columnas_fallas if fila[col] == 1]),
    axis = 1
)

tabla_fallas_multiples = fallas_multiples [['Machine_failure', 'Fallas simultáneas']].reset_index()
tabla_fallas_multiples.columns = ['ID ciclo', 'Alarma general', 'Modos de fallas detectados']

tabla_render = tabla_fallas_multiples.set_index('ID ciclo')

# Mostramos la cantidad total abajo
print(f"Total de registros con fallas simultáneas: {len(tabla_fallas_multiples)}")



Al cruzar los datos globales con las etiquetas específicas, la matemática nos arroja una diferencia neta de **34 fallas** ($373 - 339 = 34$). Sin embargo, el filtro de registros con fallas simultáneas solo detecta **24 ciclos**. 

Entonces, **Qué pasó acá??**, empecemos a investigar que pasa con esos diez datos faltantes.


In [ ]:
# Filtramos el DataFrame, cuando Machine_failure = 0 y alguna columna de fallo es = 1
dataset_filtrado_1 = df[(df['Machine_failure'] == 0) & (df[columnas_fallas].sum(axis=1) == 1)]
dataset_filtrado_1_1 = df[(df['Machine_failure'] == 1) & (df['RNF'] == 1)]

display(dataset_filtrado_1)
display(dataset_filtrado_1_1)

# Realizamos el conteo de filas
print('Hay un total de:', len(dataset_filtrado_1)+len(dataset_filtrado_1_1), 'registros donde Machine failura = 0 y RNF = 1')


In [ ]:
dataset_filtrado_2 = df[(df['Machine_failure'] == 1) & (df[columnas_fallas].sum(axis=1) == 0)]
display(dataset_filtrado_2)

# Realizamos el conteo de filas
print('Hay un total de:', len(dataset_filtrado_2), 'registros donde Machine failura = 1 y [TWF, HDF, PWF, OSF] = 0')


### Análisis de Discrepancias en los Registros de Falla

Al comparar la variable objetivo principal (`Machine_failure`) con las columnas de fallas específicas (`TWF`, `HDF`, `PWF`, `OSF`, `RNF`), se observa una inconsistencia numérica. Mientras que el sistema registra **339** paradas de máquina generales, la suma individual de las alarmas específicas asciende a **373**. 

Esta diferencia neta de **34 eventos** (373 - 339) es el resultado de tres comportamientos operativos y de recolección de datos dentro del dataset:

#### 1. Fallas Simultáneas (+24 en fallas específicas)
Se identificaron **24 ciclos de trabajo** donde la máquina experimentó dos modos de falla al mismo tiempo (por ejemplo, un pico de sobreesfuerzo mecánico que deriva en un problema de disipación de calor).

#### 2. Fallas Aleatorias u Ocultas (+19 en fallas específicas)
Existen **19 registros** clasificados como fallas aleatorias (`RNF` = 1) donde la alarma general de la máquina nunca se activó (`Machine_failure` = 0).
* **Impacto en los datos:** Al estar presentes en las causas específicas pero no sumar al conteo general, generan un excedente de **+19**.
* **Significado analítico:** La documentación técnica del dataset define las `RNF` como simulaciones de eventos ajenos al proceso de mecanizado (ej. caídas de tensión eléctrica o errores de hardware). Representan ruido estocástico que no responde a la termodinámica ni a la cinemática del equipo.

#### 3. Fallas Fantasma (-9 en fallas específicas)
Se detectaron **9 casos** donde la máquina registró una parada general (`Machine_failure` = 1), pero todos los sensores de fallas específicas (incluyendo las aleatorias) marcaron 0.
* **Impacto en los datos:** Suman 9 eventos al recuento general sin aportar ninguna causa específica, lo que resta **-9** a la diferencia neta.
* **Significado analítico:** En un entorno industrial, esto equivale a una falsa alarma del panel de control o a una parada por un factor no medido en la telemetría actual. 

#### Balance Final de la Discrepancia
La diferencia matemática queda perfectamente cerrada al cruzar estos tres fenómenos en el dataset:

* **+ 24** (Fallas múltiples que duplican la causa específica)
* **+ 19** (Fallas aleatorias no registradas en la alarma general)
* **- 9** (Falsas alarmas sin causa registrada)
* **= 34** (Diferencia exacta entre las 373 causas específicas y las 339 paradas generales).


### Visualización de Datos

1. Análisis descriptivo univariado. Seleccionamos las variables numéricas `Air Temperature [K]`, `Process Temperature [K]`, `Rotational Speed [rpm]`, `Torque_Nm`, `Tool Wear [min]` para analizar asimetría y curtosis


In [ ]:
# Primero identificamos los tipos de cada variable
df.dtypes

In [ ]:
# Nos quedamos únicamente con las variables físicas
variables_fisicas = [
    'Air_temperature_K', 
    'Process_temperature_K', 
    'Rotational_speed_rpm', 
    'Torque_Nm', 
    'Tool_wear_min'
]

Vemos en la siguiente celda el resultado de calcular la asimetría y la curtosis. La única variable que muestra una asimetría más marcada es `Rotational Speed [rpm]`. Los valores altos de curtosis indican una distribución estirada, con colas muy anchas y valores extremos.


In [ ]:
print(f"Asimetría de las variables: \n{df[variables_fisicas].skew()}\n")
print(f"Curtosis de las variables: \n{df[variables_fisicas].kurtosis()}")

2. Distribución de las variables


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for i, col in enumerate(variables_fisicas):
    sns.histplot(
        data=df, 
        x=col, 
        hue='Machine_failure', 
        kde=True, 
        stat='density',
        common_norm=False,
        palette='Set1',
        alpha=0.5,
        ax=axes[i]
    )
    
    axes[i].set_title(col, fontweight='bold')
    axes[i].set_xlabel('')
    axes[i].set_ylabel('')

axes[5].axis('off')
fig.supylabel('Density', x=0.01)

plt.tight_layout()
plt.show()

3. Boxplot de las variables


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for i, col in enumerate(variables_fisicas):
    sns.boxplot(
        data=df, 
        x='Machine_failure', 
        y=col, 
        ax=axes[i]
    )

    axes[i].set_title(col, fontweight='bold')
    axes[i].set_ylabel('')

axes[5].axis('off')

plt.tight_layout()
plt.show()

Observamos tanto en los histogramas como en los boxplot, que la distribución de variables es distinta en los casos que falló la máquina. Se observa una temperatura más alta, al igual que un torque y un tiempo de uso de herramienta mayor, a su vez que la velocidad rotacional disminuye significativamente. Esta variación en las distribuciones serán las que nos servirán para entrenar a nuestro modelo predictivo


4. Análisis de la variable categórica Type

In [ ]:
fallas_por_tipo = df.groupby('Type')[columnas_fallas].sum()

fallas_por_tipo['Total_Fallas'] = fallas_por_tipo.sum(axis=1)

fallas_por_tipo = fallas_por_tipo.sort_values(by='Total_Fallas', ascending=False)

display(fallas_por_tipo)

In [ ]:
fallas_melt = df.melt(id_vars=['Type'],
                      value_vars=columnas_fallas,
                      var_name='Modo_Falla',
                      value_name='Ocurrencia')

fallas_positivas = fallas_melt[fallas_melt['Ocurrencia'] == 1]

plt.figure(figsize=(10,6))
sns.countplot(
    data=fallas_positivas,
    x='Modo_Falla',
    hue='Type',
    palette='Set1',
    order=columnas_fallas
)

plt.title('Frecuencia de Modos de Falla según la calidad del producto (Type)', fontweight='bold')
plt.xlabel('Modos de Falla')
plt.ylabel('Cantidad de Eventos')
plt.legend(title='Tipo de Producto (L, M, H)')
plt.tight_layout()
plt.show()

### Análisis de relaciones e hipótesis físicas

Las fallas mecánicas pueden estar asociadas con la interacción entre distintas condiciones operativas. Por este motivo, estudiaremos combinaciones de variables que permitan identificar patrones que no necesariamente se observan al analizar cada característica de forma aislada.

Se proponen las siguientes hipótesis exploratorias:

1. **Diferencia de temperaturas:** analizar si la diferencia entre la temperatura del proceso y la del aire presenta patrones distintos en los registros con y sin falla.
2. **Torque y velocidad de rotación:** estudiar la relación entre la carga mecánica y la velocidad de operación.
3. **Desgaste de herramienta y torque:** explorar si los niveles de desgaste se asocian con diferentes condiciones de esfuerzo mecánico y con una mayor proporción de fallas.
4. **Combinaciones de variables:** investigar si las fallas aparecen con mayor frecuencia en determinadas regiones del espacio de características.

Estas hipótesis se contrastarán mediante visualizaciones multivariables. Los patrones observados se interpretarán como asociaciones exploratorias, sin asumir de antemano que exista una relación causal.


Vamos a crear dos variables nuevas para investigar las combinaciones sugeridas anteriormente:

- **$\Delta T$:** diferencia entre la temperatura de proceso y la temperatura del aire.
- **$P$:** potencia mecánica aproximada, calculada a partir del torque y la velocidad angular.

La potencia se calcula mediante la siguiente expresión:

$$
P = \tau \omega = \tau \frac{2\pi n}{60}
$$

Donde:

- $\tau$: torque, expresado en newton-metro (Nm).
- $\omega$: velocidad angular, expresada en radianes por segundo (rad/s).
- $n$: velocidad de rotación, expresada en revoluciones por minuto (rpm).
- $P$: potencia mecánica aproximada, expresada en watts (W).

In [ ]:
# Copia para el análisis exploratorio de relaciones físicas
df_analisis = df.copy()

# Diferencia entre temperatura de proceso y temperatura del aire
df_analisis["Temperature_difference_K"] = (
    df_analisis["Process_temperature_K"]
    - df_analisis["Air_temperature_K"]
)

# Velocidad angular en rad/s
df_analisis["Angular_speed_rad_s"] = (
    2 * np.pi * df_analisis["Rotational_speed_rpm"] / 60
)

# Potencia mecánica aproximada en watts
df_analisis["Mechanical_power_W"] = (
    df_analisis["Torque_Nm"]
    * df_analisis["Angular_speed_rad_s"]
)

display(
    df_analisis[
        [
            "Air_temperature_K",
            "Process_temperature_K",
            "Temperature_difference_K",
            "Rotational_speed_rpm",
            "Torque_Nm",
            "Mechanical_power_W",
            "Tool_wear_min",
            "Machine_failure"
        ]
    ].head()
)


Gráfico temperatura y velocidad

In [ ]:
plt.figure(figsize=(10, 6))

sns.scatterplot(
    data=df_analisis,
    x="Temperature_difference_K",
    y="Rotational_speed_rpm",
    hue="Machine_failure",
    palette={0: "steelblue", 1: "crimson"},
    alpha=0.55
)

plt.title("Diferencia de temperatura y velocidad de rotación según falla")
plt.xlabel("Diferencia de temperatura [K]")
plt.ylabel("Velocidad de rotación [rpm]")
plt.legend(title="Machine failure")
plt.tight_layout()
plt.show()

En el gráfico se puede explorar si la combinación entre diferencia de temperaturas y velocidad de rotación separa parcialmente los registros con y sin falla. La presencia de algunos puntos de falla en una región no valida por sí sola la hipótesis; hay que evaluar cuánto se solapan las clases y comprobar el patrón con métricas y datos no usados para ajustar el modelo.

Se observa un patrón clarísimo de fallas (puntos rojos 1) en dos extremos térmicos y operativos: las velocidades muy bajas (cercanas a 1250 rpm) y las zonas con alta diferencia de temperatura. Esto valida completamente su Hipótesis 1.


In [ ]:
plt.figure(figsize=(10, 6))

sns.scatterplot(
    data=df_analisis,
    x="Rotational_speed_rpm",
    y="Torque_Nm",
    hue="Machine_failure",
    size="Tool_wear_min",
    sizes=(15, 130),
    palette={0: "steelblue", 1: "crimson"},
    alpha=0.6,
    legend="brief"
)

plt.title("Torque y velocidad de rotación según falla y desgaste")
plt.xlabel("Velocidad de rotación [rpm]")
plt.ylabel("Torque [Nm]")
plt.tight_layout()
plt.show()

El gráfico permite explorar conjuntamente torque, velocidad y desgaste de herramienta. Si las fallas aparecen concentradas en ciertas regiones, eso puede orientar la ingeniería de características y la selección de modelos. No conviene afirmar que una región es “de riesgo” sin cuantificar la tasa de falla por región y considerar el bajo número de casos positivos.

Las fallas se concentran fuertemente en el extremo superior izquierdo (Torque muy alto y Velocidad baja) y algunas en el extremo inferior derecho (Velocidad muy alta con Torque bajo, combinado con un desgaste de herramienta grande por el tamaño de las burbujas).


In [ ]:
# Mapeamos la variable categórica ordinal a valores numéricos asumiendo jerarquía
df_analisis['Type_encoded'] = df_analisis['Type'].map({'L': 1, 'M': 2, 'H': 3})

# Seleccionamos las variables numéricas, la nueva ordinal codificada y el target
variables_correlacion = [
    "Air_temperature_K", "Process_temperature_K", "Temperature_difference_K",
    "Rotational_speed_rpm", "Torque_Nm", "Mechanical_power_W", "Tool_wear_min", 
    "Type_encoded", "Machine_failure"
]

df_num = df_analisis[variables_correlacion]

# Análisis de distribución y normalidad
print("--- PRUEBA DE NORMALIDAD (SHAPIRO-WILK) ---")
# Nota: Shapiro-Wilk tiene un límite de 5000 registros en algunas librerías, 
# tomamos una muestra representativa para evaluar la forma.
df_sample = df_num.sample(min(4000, len(df_num)), random_state=42)

for col in variables_correlacion:
    stat, p_value = shapiro(df_sample[col])
    print(f"Variable '{col}': p-value = {p_value:.5f}")
    if p_value > 0.05:
        print(f"  -> Sigue una distribución Normal (Acepta H0)")
    else:
        print(f"  -> NO sigue una distribución Normal (Rechaza H0 - Usar Kendall/Spearman)")


# 2. CÁLCULO DE CORRELACIONES (PEARSON VS KENDALL)

# Dado que la velocidad y el torque muestran una clara relación NO lineal (curva),
# la correlación de Pearson subestimará su fuerza. Usamos Kendall.

plt.figure(figsize=(18, 6))

# Matriz Pearson
plt.subplot(1, 2, 1)
corr_pearson = df_num.corr(method='pearson')
sns.heatmap(corr_pearson, annot=True, cmap="coolwarm", fmt=".2f", vmin=-1, vmax=1)
plt.title("Matriz de Correlación de Pearson (Relaciones Lineales)")

# Matriz Kendall
plt.subplot(1, 2, 2)
corr_kendall = df_num.corr(method='kendall')
sns.heatmap(corr_kendall, annot=True, cmap="coolwarm", fmt=".2f", vmin=-1, vmax=1)
plt.title("Matriz de Correlación de Kendall (Relaciones Monótonas / No Lineales)")

plt.tight_layout()
plt.show()


In [26]:
# Eliminamos columnas que generarían Target Leakage (filtración de datos)
# No podemos predecir si la máquina falló usando los sensores de falla que se activan durante el fallo.
df_final = df_analisis.drop(columns=['TWF', 'HDF', 'PWF', 'OSF', 'RNF'])

# Eliminamos la columna categórica original porque ya usamos la codificada
df_final = df_final.drop(columns=['Type'])

# Exportamos a CSV para el notebook de Machine Learning
df_final.to_csv('ai4i2020_model_ready.csv', index=False)
print("Dataset exportado exitosamente para modelado.")

Dataset exportado exitosamente para modelado.
